In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv(r"D:\Practice\Numpy\Practice\Dataset\4train.csv")

df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


In [ ]:
#1
"""
1. Create a copy of the Age column as a NumPy array. 
2. Replace missing Age values with the median Age of passengers in the same Pclass. 
3. Store the filled values in a new DataFrame column called AgeFilled. 
4. Calculate the mean of AgeFilled for each Pclass using transform(). 
5. Display the 10 passengers whose age differs most from their class average. 

Required columns: Name, Pclass, AgeFilled, ClassAvgAge, AgeDifference
"""

df['AgeFilled'] = df['Age'].fillna(df.groupby('Pclass')['Age'].transform('median'))

df['ClassAvgAge'] = df.groupby('Pclass')['AgeFilled'].transform('mean')

df['AgeDifference'] = (df['AgeFilled'] - df['ClassAvgAge']).abs()

show = df[['Name','Pclass','AgeFilled','ClassAvgAge','AgeDifference']].sort_values('AgeDifference', ascending = False).head(10)

print(show)

                                     Name  Pclass  AgeFilled  ClassAvgAge  \
851                   Svensson, Mr. Johan       3      74.00    24.824684   
116                  Connors, Mr. Patrick       3      70.50    24.824684   
630  Barkworth, Mr. Algernon Henry Wilson       1      80.00    38.062130   
280                      Duane, Mr. Frank       3      65.00    24.824684   
672           Mitchell, Mr. Henry Michael       2      70.00    29.825163   
483                Turkula, Mrs. (Hedwig)       3      63.00    24.824684   
305        Allison, Master. Hudson Trevor       1       0.92    38.062130   
326             Nysveen, Mr. Johan Hansen       3      61.00    24.824684   
33                  Wheadon, Mr. Edward H       2      66.00    29.825163   
297          Allison, Miss. Helen Loraine       1       2.00    38.062130   

     AgeDifference  
851      49.175316  
116      45.675316  
630      41.937870  
280      40.175316  
672      40.174837  
483      38.175316  
305  

In [15]:
#2
"""
1. Convert Fare into a NumPy array copy. 
2. Replace missing fares with the median Fare. 
3. Apply min-max normalization : normalizedFare= 𝑥−min(𝑥) / max(𝑥)−min(𝑥) 
4. Print the minimum, maximum, and mean of the normalized array. 
5. Confirm that the original df['Fare'] column was not modified.
"""

fare = df['Fare'].to_numpy().copy()

fare = np.where(np.isnan(fare), np.nanmedian(fare), fare)

normalizedFare = ((fare - fare.min()) /(fare.max() - fare.min()))

print(f"Minimum of normalized array : {normalizedFare.min()}")
print(f"Maximum of normalized array : {normalizedFare.max()}")
print(f"Mean of normalized array : {normalizedFare.mean()}")

Minimum of normalized array : 0.0
Maximum of normalized array : 1.0
Mean of normalized array : 0.06285842768394742


In [20]:
#3
"""
1. Create FamilySize = SibSp + Parch + 1. 
2. Create a FamilyCategory column: 
                    o Alone: FamilySize = 1 
                    o Small: FamilySize = 2–4 
                    o Large: FamilySize > 4 

3. Calculate the survival rate percentage for each category. 
4. Sort the results from highest to lowest survival rate. 
5. Display the number of passengers in each category. 
"""

df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

df['FamilyCategory'] = np.where(df['FamilySize'] == 1, 'Alone',
                                    np.where(df['FamilySize'] <= 4,'Small','Large'))

show = (df.groupby('FamilyCategory').agg(
          PassengerCount=('Survived', 'count'),
          SurvivalRate=('Survived', 'mean')
      )
)

show['SurvivalRate'] = show['SurvivalRate'] * 100

show = show.sort_values('SurvivalRate', ascending=False)

print(show)

                PassengerCount  SurvivalRate
FamilyCategory                              
Small                      292     57.876712
Alone                      537     30.353818
Large                       62     16.129032


In [27]:
#4
"""
Use NumPy to detect fare outliers using the IQR method. 
1. Calculate Q1, Q3, and IQR for Fare. 
2. Calculate the lower and upper bounds. 
3. Idetify outliers outside those bounds. 
4. Print the number of outliers and their percentage of the dataset. 
5. Display the 10 highest-fare outliers with Name, Pclass, and Fare.
"""

q1 = np.percentile(fare, 25)
q3 = np.percentile(fare, 75)

iqr = q3 - q1

lower = q1 - (1.5 * iqr)
upper = q3 + (1.5 * iqr)

outlier_mask = (fare < lower) | (fare > upper)

count = np.sum(outlier_mask)
percentage = (count / len(fare)) * 100

print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Number of outliers:", count)
print("Percentage of outliers:", percentage)

Q1: 7.9104
Q3: 31.0
IQR: 23.0896
Lower bound: -26.724
Upper bound: 65.6344
Number of outliers: 116
Percentage of outliers: 13.019079685746352


In [ ]:
#5
"""

"""